## Libraries

In [7]:
# weather data 
import meteostat as ms

# from src
from src.spark_config import initialize_spark 
from src.helper_functions import get_paths, get_coordinates 

# spark biblioteker fra labene
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, lit, when, count, sum as spark_sum, avg,
    min as spark_min, max as spark_max, round as spark_round,
    desc, asc, to_timestamp, to_date, hour, dayofweek,
    lower, upper, trim, concat, coalesce, explode, split,
    countDistinct, collect_list, first, input_file_name,
    regexp_extract
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, BooleanType, LongType, ArrayType
)

# other libraries
import pandas as pd
import numpy as np
import regex as re
from datetime import datetime, timedelta
from pathlib import Path

# specify java version
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

## Initialize

In [3]:
# set spark session
spark = initialize_spark()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/20 14:52:14 WARN Utils: Your hostname, esoey-Laptop, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/20 14:52:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/esoey/DAT535/DAT535_FootballStats/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/20 14:52:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 4.3.0
Application ID: local-1795182736944
Spark UI: http://10.255.255.254:4040


## Paths

In [4]:
STATSBOMB_PATHS = get_paths()

In [5]:
competitions_schema = StructType([
        StructField("competition_gender", StringType()),
        StructField("competition_id", LongType()), 
        StructField("competition_international", BooleanType()),
        StructField("competition_name", StringType()),
        StructField("competition_youth", BooleanType()),
        StructField("country_name", StringType()),
        StructField("match_available", StringType()),
        StructField("match_available_360", StringType()),
        StructField("match_updated", StringType()),
        StructField("match_updated_360", StringType()),
        StructField("season_id", LongType()),
        StructField("season_name", StringType())
    ])
competitions_df = spark.read.schema(competitions_schema).json(STATSBOMB_PATHS["COMPETITIONS"], multiLine=True)

In [8]:
matches_schema = StructType([
        StructField("away_score", LongType()),
        StructField("away_team", StructType([
            StructField("away_team_gender", StringType()),
            StructField("away_team_group", StringType()),
            StructField("away_team_id", LongType()),
            StructField("away_team_name", StringType()),
            StructField("country", StructType([
                StructField("id", LongType()),
                StructField("name", StringType()),
            ])),
            StructField("managers", ArrayType(StructType([
                StructField("country", StructType([
                    StructField("id", LongType()),
                    StructField("name", StringType()),
                ])),
                StructField("dob", StringType()),
                StructField("id", LongType()),
                StructField("name", StringType()),
                StructField("nickname", StringType()),
            ]))),
        ])),
        StructField("competition", StructType([
            StructField("competition_id", LongType()),
            StructField("competition_name", StringType()),
            StructField("country_name", StringType()),
        ])),
        StructField("competition_stage", StructType([
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("home_score", LongType()),
        StructField("home_team", StructType([
            StructField("country", StructType([
                StructField("id", LongType()),
                StructField("name", StringType())
            ])),
            StructField("home_team_gender", StringType()),
            StructField("home_team_group", StringType()),
            StructField("home_team_id", LongType()),
            StructField("home_team_name", StringType()),
            StructField("managers", ArrayType(StructType([
                StructField("country", StructType([
                    StructField("id", LongType()),
                    StructField("name", StringType()),
                ])),
                StructField("dob", StringType()),
                StructField("id", LongType()),
                StructField("name", StringType()),
                StructField("nickname", StringType()),
            ]))),
        ])),
        StructField("kick_off", StringType()),
        StructField("last_updated", StringType()),
        StructField("last_updated_360", StringType()),
        StructField("match_date", StringType()),
        StructField("match_id", LongType()),
        StructField("match_status", StringType()),
        StructField("match_status_360", StringType()),
        StructField("match_week", LongType()),
        StructField("metadata", StructType([
            StructField("data_version", StringType()),
            StructField("shot_fidelity_version", StringType()),
            StructField("xy_fidelity_version", StringType()),
        ])),
        StructField("referee", StructType([
            StructField("country", StructType([
                StructField("id", LongType()),
                StructField("name", StringType()),
            ])),
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
        StructField("season", StructType([
            StructField("season_id", LongType()),
            StructField("season_name", StringType()),
        ])),
        StructField("stadium", StructType([
            StructField("country", StructType([
                StructField("id", LongType()),
                StructField("name", StringType()),
            ])),
            StructField("id", LongType()),
            StructField("name", StringType()),
        ])),
    ])
matches_df = spark.read.schema(matches_schema).json(STATSBOMB_PATHS["MATCHES"], multiLine=True, recursiveFileLookup=True)

In [9]:
print(matches_df.count())

3961


In [ ]:
events_df = spark.read.json(STATSBOMB_PATHS["EVENTS"], multiLine=True)

In [ ]:
print(competitions_df.printSchema())